In [2]:
import os
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, confusion_matrix

## Load Data

In [3]:
def read_fasta(path):
    headers, sequences = [], []
    h, buf = None, []
    
    with open(path, 'r') as f:
        for line in f:
            if not line: 
                continue
            if line[0] == '>':
                if h is not None:
                    sequences.append(''.join(buf).upper())
                    buf = []
                h = line[1:].strip()
                headers.append(h)
            else:
                buf.append(line.strip())
        if h is not None:
            sequences.append(''.join(buf).upper())
            
    return headers, sequences

def load_labels(path):
    label_dict = {}
    with open(path, 'r') as f:
        for line in f:
            if not line:
                continue
            parts = line.strip().split('\t')
            header = parts[0][1:]
            if parts[1] == "None":
                label = 0.0
            else:
                label = 1.0
            label_dict[header] = label
    return label_dict

## Data Processing

In [4]:
# mapping ACGT to 0123
# mapping acgt to 0123
# mapping N and others to T

ENCODE = np.full(256, 4, dtype=np.int64)
for ch, idx in zip(b"ACGTNacgtn", [0, 1, 2, 3, 3, 0, 1, 2, 3, 3]):
    ENCODE[ch] = idx

# reverse complement ACGTN -> TGCAN -> 32104
REV_COMP = torch.tensor([3, 2, 1, 0, 4]) 

In [5]:
class SeqDataset(Dataset):
    def __init__(self, headers, sequences, labels):
        self.headers = headers
        self.sequences = sequences
        self.labels = np.asarray(labels, dtype=np.float32)

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return self.headers[idx], self.sequences[idx].encode('ascii', 'ignore'), self.labels[idx]

In [6]:
def collate_pad(batch):
    headers, seq_bytes, labels = zip(*batch)
    idx_list = [ENCODE[np.frombuffer(sb, dtype=np.uint8)] for sb in seq_bytes]
    max_length = max(x.size for x in idx_list)
    batch_size = len(idx_list)
    
    X = torch.zeros(batch_size, 5, max_length, dtype=torch.float32)   # channels: A, C, G, T, N
    mask = torch.zeros(batch_size, max_length, dtype=torch.bool)      # padding mask
    
    for i, idx_np in enumerate(idx_list):
        length = idx_np.size
        idx = torch.from_numpy(idx_np).long()
        X[i, idx, torch.arange(length)] = 1.0
        mask[i, :length] = (idx != 4)   # invalid positions: N or everything beyond the length
    
    Y = torch.tensor(labels, dtype=torch.float32)
    return headers, X, mask, Y

## CNN Network

In [7]:
# Reverse complement convolution layer
# input: one-hot (B, 5, L)
# output: RC-invariant feature map (B, C, L)

def rc_kernel(k: torch.Tensor):
    # Reverse complement the convolution kernel
    # k: (out_channels, in_channels, length)
    k_flip = k.flip(-1)
    return k_flip.index_select(1, REV_COMP.to(k.device))

class RCFirstConv1d(nn.Module):
    
    def __init__(self, out_channels, kernel_size=15, dilation=1, bias=True, dropout=0.1):
        super().__init__()
        assert kernel_size % 2 == 1
        pad = (kernel_size // 2) * dilation
        self.conv = nn.Conv1d(5, out_channels, kernel_size, padding=pad, dilation=dilation, bias=bias)
        self.batch_norm = nn.BatchNorm1d(out_channels)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        
        y1 = self.conv(x)             # (B, C, L)
        
        x_rc = x.flip(-1)[:, REV_COMP.to(x.device), :]
        y2 = self.conv(x_rc)          # (B, C, L)
        y2 = y2.flip(-1)
        
        y = torch.max(y1, y2)
        y = self.batch_norm(y)
        y = F.gelu(y)
        y = self.dropout(y)
        
        return y 

In [8]:
class ConvBlock(nn.Module):
    def __init__(self, c_in, c_out, kernel_size=9, dilation=1, dropout=0.1):
        super().__init__()
        pad = (kernel_size // 2) * dilation
        self.conv = nn.Conv1d(c_in, c_out, kernel_size, padding=pad, dilation=dilation)
        self.batch_norm = nn.BatchNorm1d(c_out)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        return self.dropout(F.gelu(self.batch_norm(self.conv(x))))

class MaskedMaxPool1d(nn.Module):
    def __init__(self, kernel_size=2, stride=2):
        super().__init__()
        self.kernel_size, self.stride = kernel_size, stride

    def forward(self, x, mask):  # x:(B, C, L), mask:(B, L)
        m = mask.unsqueeze(1)
        x_masked = x.masked_fill(~m, float('-inf'))
        y = F.max_pool1d(x_masked, self.kernel_size, self.stride)
        m_pooled = F.max_pool1d(m.float(), self.kernel_size, self.stride) > 0
        y = y.masked_fill(~m_pooled, 0.0)
        return y, m_pooled.squeeze(1)


In [9]:
class RCInputInvariantCNN(nn.Module):
    def __init__(self, width=64, motif_kernel = (9, 15, 21, ), context_kernel = 9, context_dilation = (1, 2, ), dropout=0.1):
        super().__init__()

        self.motif_conv_blocks = nn.ModuleList(
            RCFirstConv1d(width, kernel_size=ker, dilation=1, dropout=0.1)
            for ker in motif_kernel
        )
        dim = width * len(motif_kernel)

        self.context_blocks = nn.ModuleList()
        c_in = dim
        for dil in context_dilation:
            self.context_blocks.append(
                ConvBlock(c_in, width, kernel_size=context_kernel, dilation=dil, dropout=dropout)
            )
            c_in = width

        self.pool = MaskedMaxPool1d(kernel_size=2, stride=2)
        head_in_features = c_in

        self.head = nn.Sequential(
            nn.Linear(head_in_features, width),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(width, 1),
        )

    @staticmethod
    def masked_avg_pool(z, mask):
        if mask is None:
            return z.mean(-1)
        m = mask.unsqueeze(1).float()
        return (z * m).sum(-1) / m.sum(-1).clamp_min(1.0)

    def forward(self, x, mask):
        motif_outputs = [motif_conv(x) for motif_conv in self.motif_conv_blocks]
        z = torch.cat(motif_outputs, dim=1)

        for block in self.context_blocks:
            try:
                z = block(z) + z
            except RuntimeError:
                z = block(z)
            z, mask = self.pool(z, mask)

        z_pooled_avg = self.masked_avg_pool(z, mask)
        out = self.head(z_pooled_avg).squeeze(-1)
        
        return out

## Train & Eval

In [10]:
def resolve_device(requested=None):
    """Return the best available accelerator as a torch.device."""
    if requested is not None:
        return torch.device(requested)
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

In [11]:
with torch.no_grad():
    device = resolve_device()
    model = RCInputInvariantCNN(width=16).to(device)
    X = torch.randn(3, 5, 64, device=device)
    mask = torch.zeros(3, 64, dtype=torch.bool, device=device)
    mask[0, :120] = True
    mask[1, :100] = True
    mask[2, :80] = True
    out = model(X, mask)
    print('Device:', device)
    print('Output shape:', out.shape)

Device: mps
Output shape: torch.Size([3])


In [12]:
def save_torch(data, path, filename): 
    os.makedirs(path, exist_ok=True)
    if os.path.exists("%s/%s.pt"%(path, filename)):
        n = 1
        while n <= 100:
            if os.path.exists("%s/%s_legend_%s.pt"%(path, filename, n)):
                n += 1
            else:
                torch.save(data, "%s/%s_legend_%s.pt"%(path, filename, n))
                break
    else:
        torch.save(data, "%s/%s.pt"%(path, filename))

In [13]:
def prepare_data(fasta_path, label_path, subset_size=None, random_state=42):
    headers, sequences = read_fasta(fasta_path)
    label_dict = load_labels(label_path)
    labels = [label_dict[h] for h in headers]
    
    total_sequences = len(sequences)
    if subset_size is not None and subset_size < total_sequences:
        rng = np.random.default_rng(random_state)
        perm = rng.permutation(total_sequences)
        sel_idx = np.sort(perm[:subset_size])
        headers = [headers[i] for i in sel_idx]
        sequences = [sequences[i] for i in sel_idx]
        labels = [labels[i] for i in sel_idx]
        print(f"Using random subset of {len(headers)} sequences (from {total_sequences}) with random_state={random_state}.")
    else:
        print(f"Using all {total_sequences} sequences for training.")

    # ---- Species-grouped 60 / 20 / 20 split (added by split-fix) ----
    # Header format: >{family_id}-{species_code}#{class}/{superfamily}
    # family_id may contain '-', so split from the right on the last '-'.
    def _species_from_header(h):
        return h.lstrip(">").split("#", 1)[0].rsplit("-", 1)[1]
    species = np.array([_species_from_header(h) for h in headers])
    y_arr = np.asarray(labels)
    
    # Outer 80/20 trainval/test split on species.
    gss_outer = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    idx_trainval, idx_te = next(gss_outer.split(
        np.arange(len(sequences)), y=y_arr, groups=species
    ))
    # Inner 75/25 train/val split (= 60/20 of total) on species again,
    # disjoint from the test species above.
    gss_inner = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
    inner_tr, inner_val = next(gss_inner.split(
        idx_trainval, y=y_arr[idx_trainval], groups=species[idx_trainval]
    ))
    idx_tr = idx_trainval[inner_tr]
    idx_val = idx_trainval[inner_val]
    
    # Hard guard: train / val / test species must be pairwise disjoint.
    sp_tr, sp_val, sp_te = set(species[idx_tr]), set(species[idx_val]), set(species[idx_te])
    assert not (sp_tr & sp_val), f"Species leak train/val: {sorted(sp_tr & sp_val)}"
    assert not (sp_tr & sp_te), f"Species leak train/test: {sorted(sp_tr & sp_te)}"
    assert not (sp_val & sp_te), f"Species leak val/test: {sorted(sp_val & sp_te)}"
    print(f"  train: {len(idx_tr)} seqs / {len(sp_tr)} species")
    print(f"  val:   {len(idx_val)} seqs / {len(sp_val)} species")
    print(f"  test:  {len(idx_te)} seqs / {len(sp_te)} species")
    
    ds_tr = SeqDataset([headers[i] for i in idx_tr], [sequences[i] for i in idx_tr], [labels[i] for i in idx_tr])
    ds_val = SeqDataset([headers[i] for i in idx_val], [sequences[i] for i in idx_val], [labels[i] for i in idx_val])
    ds_te = SeqDataset([headers[i] for i in idx_te], [sequences[i] for i in idx_te], [labels[i] for i in idx_te])
    
    return ds_tr, ds_val, ds_te

In [14]:
def train_one_epoch(model, loader, opt, loss_fn, device, ep, epochs):
    model.train()
    total_loss = 0.0
    pbar = tqdm(loader, desc=f"Epoch {ep}/{epochs}", leave=False)
    for _, X, mask, Y in pbar:
        X, mask, Y = X.to(device), mask.to(device), Y.to(device)
        opt.zero_grad()
        logits = model(X, mask)
        loss = loss_fn(logits, Y)
        loss.backward()
        opt.step()
        total_loss += loss.item() * X.size(0)
        pbar.set_postfix(loss=f"{loss.item():.4f}")
    avg_loss = total_loss / len(loader.dataset)
    return avg_loss

def evaluate(model, loader, loss_fn, device, dataset):
    model.eval()
    val_running = 0.0
    all_logits = []
    all_labels = []
    with torch.no_grad():
        for _, X, mask, Y in tqdm(loader, desc="Val", leave=False):
            X, mask = X.to(device, dtype=torch.float32), mask.to(device)
            Y, logits = Y.to(device, dtype=torch.float32), model(X, mask)
            loss = loss_fn(logits, Y)
            val_running += loss.item() * X.size(0)
            all_logits.append(torch.sigmoid(logits).detach().cpu().numpy())
            all_labels.append(Y.detach().cpu().numpy())
    all_logits = np.concatenate(all_logits) if all_logits else np.array([])
    all_labels = np.concatenate(all_labels) if all_labels else np.array([])
    val_loss = val_running / len(dataset) if len(dataset) else float("nan")
    
    try:
        auroc = roc_auc_score(all_labels, all_logits)
        auprc = average_precision_score(all_labels, all_logits)
    except ValueError:
        auroc, auprc = float("nan"), float("nan")
    return auroc, auprc, val_loss, all_logits, all_labels

In [15]:
resolve_device(device)

device(type='mps')

In [ ]:
def run_train(fasta_path, label_path, batch_size=8, epochs=5, lr=1e-3, device=None, patience=10, subset_size=None, random_state=42, trial=False, pos_weight=None):
    device = resolve_device(device)
    print(f"Using device: {device}")

    ds_tr, ds_val, ds_te = prepare_data( fasta_path, label_path, subset_size=subset_size, random_state=random_state)
    loader_tr = DataLoader(ds_tr, batch_size=batch_size, shuffle=True, num_workers=0, collate_fn=collate_pad)
    loader_val = DataLoader(ds_val, batch_size=batch_size, shuffle=False, num_workers=0, collate_fn=collate_pad)
    loader_te = DataLoader(ds_te, batch_size=batch_size, shuffle=False, num_workers=0, collate_fn=collate_pad)

    # The dataset is highly imbalanced; compute pos_weight if not provided
    if pos_weight is None:
        train_labels = torch.tensor(ds_tr.labels, dtype=torch.float32)
        pos_count = float(train_labels.sum().item())
        neg_count = float(train_labels.numel() - pos_count)
        if pos_count == 0:
            pos_weight_value = 1.0
        else:
            pos_weight_value = max(neg_count / pos_count, 1.0)
        print(f"Computed pos_weight={pos_weight_value:.4f} from dataset (pos={pos_count:.0f}, neg={neg_count:.0f})")
        pos_weight_tensor = torch.tensor(pos_weight_value, device=device)
    else:
        if isinstance(pos_weight, torch.Tensor):
            pos_weight_tensor = pos_weight.to(device)
            pos_weight_value = float(pos_weight_tensor.item())
        else:
            pos_weight_value = float(pos_weight)
            pos_weight_tensor = torch.tensor(pos_weight_value, device=device)
        print(f"Using provided pos_weight={pos_weight_value:.4f}")

    model = RCInputInvariantCNN().to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)

    print("Starting training...")

    history = {"train_loss": [], "val_loss": [], "val_auroc": [], "val_auprc": []}
    best_state, best_epoch = None, None
    best_metrics = {"auroc": -float("inf"), "auprc": float("nan"), "val_loss": float("inf")}
    best_scores, best_labels = None, None
    last_scores, last_labels = None, None
    last_val_loss = None
    last_auroc, last_auprc = None, None
    bad = 0
    patience = patience if patience is not None else epochs + 1

    for ep in range(1, epochs + 1):
        train_loss = train_one_epoch(model, loader_tr, opt, loss_fn, device, ep, epochs)

        auroc, auprc, val_loss, Ps, Ys = evaluate(model, loader_val, loss_fn, device, ds_val)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_auroc"].append(auroc)
        history["val_auprc"].append(auprc)

        print(f"Epoch {ep}: train loss {train_loss:.4f} | Val loss {val_loss:.4f} | Val AUROC {auroc:.4f} | AUPRC {auprc:.4f}")

        last_scores, last_labels = Ps, Ys
        last_val_loss, last_auroc, last_auprc = val_loss, auroc, auprc

        improved = auroc > best_metrics["auroc"] + 1e-4
        if improved:
            best_metrics = {"auroc": auroc, "auprc": auprc, "val_loss": val_loss}
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            best_scores, best_labels, best_epoch = Ps, Ys, ep
            bad = 0
            torch.save(best_state, "rc_cnn_latest.pt")
        else:
            bad += 1
            if bad >= patience:
                print("Early stopping.")
                break
        if auroc == 1:
            print("Early stopping.")
            break

    if best_state is not None:
        if not trial:
            save_torch(best_state, ".", "rc_cnn_best")
            model.load_state_dict(best_state)
            model.to(device)
    else:
        best_scores, best_labels = last_scores, last_labels
        best_epoch = best_epoch if best_epoch is not None else len(history["train_loss"])
        best_metrics = {
            "auroc": last_auroc,
            "auprc": last_auprc,
            "val_loss": last_val_loss,
        }

    auroc, auprc, te_loss, te_scores, te_labels = evaluate(model, loader_te, loss_fn, device, ds_te)
    print(f"Test set performance: Loss {te_loss:.4f} | AUROC {auroc:.4f} | AUPRC {auprc:.4f}")

    results = {
        "model": model,
        "history": history,
        "metrics": {
            "best_epoch": best_epoch,
            "best_auroc": best_metrics["auroc"],
            "best_auprc": best_metrics["auprc"],
            "best_val_loss": best_metrics["val_loss"],
            "test_auroc": auroc,
            "test_auprc": auprc,
            "test_loss": te_loss,
        },
        "roc": {
            "labels": te_labels,
            "scores": te_scores,
            "best_labels": best_labels,
            "best_scores": best_scores,
        },
        "device": str(device),
        "pos_weight": pos_weight_value,
    }
    return results

In [17]:
fasta_path = '../../../data/vgp/all_vgp_tes.fa'
label_path = '../../../data/vgp/20260120_features_sf'
subset_size = 6000  # set to None to use the full dataset
subset_seed = 42

headers, sequences = read_fasta(fasta_path)
label_dict = load_labels(label_path)
labels = [label_dict[h] for h in headers]

In [18]:
# Long-running cell: executes full training
results = run_train(fasta_path, label_path, epochs=20, patience=None, subset_size=subset_size, random_state=subset_seed)

Using device: mps
Using random subset of 6000 sequences (from 135751) with random_state=42.
  train: 3614 seqs / 279 species
  val:   1184 seqs / 93 species
  test:  1202 seqs / 93 species
Computed pos_weight=1.7051 from dataset (pos=1336, neg=2278)
Starting training...


Epoch 1/20:   0%|          | 0/452 [00:00<?, ?it/s]

Val:   0%|          | 0/148 [00:00<?, ?it/s]

Epoch 1: train loss 0.8030 | Val loss 0.7662 | Val AUROC 0.7228 | AUPRC 0.5222


RuntimeError: Parent directory vgp_model_data_tpase does not exist.

## Plot the results

In [ ]:
def plot_training_curves(history):
    epochs = np.arange(1, len(history.get("train_loss", [])) + 1)
    if len(epochs) == 0:
        raise ValueError("History is empty; run training first.")
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(epochs, history["train_loss"], label="Train loss", marker="o")
    axes[0].plot(epochs, history["val_loss"], label="Val loss", marker="o")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Binary cross-entropy")
    axes[0].set_title("Loss curves")
    axes[0].legend()
    axes[1].plot(epochs, history["val_auroc"], label="Val AUROC", marker="o")
    axes[1].plot(epochs, history["val_auprc"], label="Val AUPRC", marker="o")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Score")
    axes[1].set_ylim(0, 1)
    axes[1].set_title("Validation metrics")
    axes[1].legend()
    fig.tight_layout()
    return axes

def plot_roc(labels, scores):
    if labels is None or scores is None:
        raise ValueError("ROC data is missing; ensure training completed successfully.")
    fpr, tpr, _ = roc_curve(labels, scores)
    auc = roc_auc_score(labels, scores)
    plt.figure(figsize=(6, 6))
    plt.plot(fpr, tpr, label=f"ROC (AUC={auc:.4f})")
    plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Chance")
    plt.xlabel("False positive rate")
    plt.ylabel("True positive rate")
    plt.title("Validation ROC curve")
    plt.legend(loc="lower right")
    plt.grid(True)

In [ ]:
history = results["history"]
plot_training_curves(history)

In [ ]:
roc_data = results["roc"]
plot_roc(roc_data["labels"], roc_data["scores"])

In [ ]:
headers, seqs = read_fasta(fasta_path)

ds_tr, ds_val, ds_te = prepare_data(fasta_path, label_path, subset_size=6000, random_state=42)
loader = DataLoader(ds_te, batch_size=8, shuffle=False, num_workers=0, collate_fn=collate_pad)

all_headers, all_probs, all_logits = [], [], []

model.eval()
with torch.no_grad():
    for hdr_batch, X, mask, Y in tqdm(loader, desc="Val", leave=False):
        X = X.to(device, dtype=torch.float32)
        mask = mask.to(device)
        logits = model(X, mask)        # (B,) for binary or (B, n_classes) for multi
        probs = torch.sigmoid(logits)  # or softmax for multi
        all_logits.append(logits.cpu().numpy())
        all_headers.extend(hdr_batch)
        all_probs.append(probs.cpu().numpy())

all_probs = np.concatenate(all_probs)
all_logits = np.concatenate(all_logits)

# Example: show top 10 “most positive” sequences
top_idx = np.argsort(-all_probs)[:10]
for i in top_idx:
    print(all_headers[i], all_probs[i], )

In [ ]:
auroc, auprc, te_loss, te_scores, te_labels = evaluate(model, loader, nn.BCEWithLogitsLoss(), device, ds_te)
print(f"Test set performance: Loss {te_loss:.4f} | AUROC {auroc:.4f} | AUPRC {auprc:.4f}")

In [ ]:
model.eval()
all_logits = []
with torch.no_grad():
    for _, X, mask, Y in tqdm(loader, desc="Val", leave=False):
        X = X.to(device, dtype=torch.float32)
        mask = mask.to(device)
        logits = model(X, mask)
        all_logits.append(logits.cpu().numpy())
all_logits = np.concatenate(all_logits)
print("logits mean/std/min/max:",
      all_logits.mean(), all_logits.std(),
      all_logits.min(), all_logits.max())

In [ ]:
ds_tr

In [ ]:
label_dict = load_labels(label_path)
labels = [label_dict[h] for h in headers]

for i in range(30):
    print(all_headers[i], all_probs[i], labels[i])

In [ ]:
min(all_probs), max(all_probs)

In [ ]:
headers, sequences = read_fasta(fasta_path)
label_dict = load_labels(label_path)
labels = [label_dict[h] for h in headers]

total_sequences = len(sequences)
if subset_size is not None and subset_size < total_sequences:
    rng = np.random.default_rng(42)
    perm = rng.permutation(total_sequences)
    sel_idx = np.sort(perm[:subset_size])
    headers = [headers[i] for i in sel_idx]
    sequences = [sequences[i] for i in sel_idx]
    labels = [labels[i] for i in sel_idx]
    print(f"Using random subset of {len(headers)} sequences (from {total_sequences}) with random_state={42}.")
else:
    print(f"Using all {total_sequences} sequences for training.")

# ---- Species-grouped 60 / 20 / 20 split (added by split-fix) ----
def _species_from_header(h):
    return h.lstrip(">").split("#", 1)[0].rsplit("-", 1)[1]
species = np.array([_species_from_header(h) for h in headers])
y_arr = np.asarray(labels)

gss_outer = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
idx_trainval, idx_te = next(gss_outer.split(
    np.arange(len(sequences)), y=y_arr, groups=species
))
gss_inner = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
inner_tr, inner_val = next(gss_inner.split(
    idx_trainval, y=y_arr[idx_trainval], groups=species[idx_trainval]
))
idx_tr = idx_trainval[inner_tr]
idx_val = idx_trainval[inner_val]

sp_tr, sp_val, sp_te = set(species[idx_tr]), set(species[idx_val]), set(species[idx_te])
assert not (sp_tr & sp_val), f"Species leak train/val: {sorted(sp_tr & sp_val)}"
assert not (sp_tr & sp_te), f"Species leak train/test: {sorted(sp_tr & sp_te)}"
assert not (sp_val & sp_te), f"Species leak val/test: {sorted(sp_val & sp_te)}"
print(f"  train: {len(idx_tr)} seqs / {len(sp_tr)} species")
print(f"  val:   {len(idx_val)} seqs / {len(sp_val)} species")
print(f"  test:  {len(idx_te)} seqs / {len(sp_te)} species")

ds_tr = SeqDataset([headers[i] for i in idx_tr], [sequences[i] for i in idx_tr], [labels[i] for i in idx_tr])
ds_val = SeqDataset([headers[i] for i in idx_val], [sequences[i] for i in idx_val], [labels[i] for i in idx_val])
ds_te = SeqDataset([headers[i] for i in idx_te], [sequences[i] for i in idx_te], [labels[i] for i in idx_te])

In [ ]:
len(label_dict)

In [ ]:
len(labels), sum(labels)